# Interpretable PyTorch neural networks for severe wasting

Two Neural Additive Models (NAMs) are fitted with the same predictor sets as `models.ipynb`: all children aged 0–4 (17 predictors) and children aged 0–1/under 2 (28 predictors). The target is `severe_wasting`. PSU-grouped stratified 5-fold cross-validation is used. A NAM is interpretable because its logit is the sum of one learned contribution per original variable.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

RANDOM_STATE=42
N_SPLITS=5
EPOCHS=60
HIDDEN=4
WEIGHT_DECAY=0.05
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(2)

In [ ]:
data=pd.read_stata('MICS-child-ISRC.dta',convert_categoricals=False)
data=data.dropna(subset=['hh1','chweight','strata','severe_wasting']).copy()
data=data[data['chweight']>0].reset_index(drop=True)
features_all=['ub2','female','urban','division','wealth','edu_level','wb4','cm11','mother_married','water_cat','toilet_cat','fuel_cat','electricity','insured','diarrhoea_2w','cough_2w','breathing_2w']
features_infant=features_all+['dds','fg_grain','fg_legume','fg_dairy','fg_flesh','fg_egg','fg_vita','fg_ofv','bf_status','bottle','anc_cat']
numeric_all=['ub2','wb4','cm11']
numeric_infant=numeric_all+['dds']
categorical_all=[c for c in features_all if c not in numeric_all]
categorical_infant=[c for c in features_infant if c not in numeric_infant]
df_all=data.copy()
df_infant=data[data['young']==1].copy()
print(f'All children: {len(df_all):,}; cases: {int(df_all.severe_wasting.sum())}; predictors: {len(features_all)}')
print(f'Ages 0-1: {len(df_infant):,}; cases: {int(df_infant.severe_wasting.sum())}; predictors: {len(features_infant)}')

## Neural additive architecture

For predictors x_j, the model is `logit(P(severe_wasting=1)) = bias + sum_j f_j(x_j)`. Continuous variables use independent 1→4→1 tanh subnetworks. Categorical variables use a learned scalar contribution per category. No cross-feature interactions are hidden inside the model, so each variable's contribution can be inspected directly.

In [ ]:
class NeuralAdditiveModel(nn.Module):
    def __init__(self,n_numeric,category_sizes,hidden=4):
        super().__init__()
        self.numeric_nets=nn.ModuleList([nn.Sequential(nn.Linear(1,hidden),nn.Tanh(),nn.Linear(hidden,1)) for _ in range(n_numeric)])
        self.category_effects=nn.ModuleList([nn.Embedding(size,1) for size in category_sizes])
        self.bias=nn.Parameter(torch.zeros(1))
    def forward(self,x_numeric,x_categorical,return_contributions=False):
        parts=[net(x_numeric[:,j:j+1]) for j,net in enumerate(self.numeric_nets)]
        parts += [emb(x_categorical[:,j]) for j,emb in enumerate(self.category_effects)]
        contributions=torch.cat(parts,dim=1)
        logits=self.bias+contributions.sum(dim=1)
        return (logits,contributions) if return_contributions else logits

def prepare_fold(train,test,numeric_cols,categorical_cols):
    med=train[numeric_cols].median()
    scaler=StandardScaler().fit(train[numeric_cols].fillna(med))
    xn_tr=scaler.transform(train[numeric_cols].fillna(med)).astype('float32')
    xn_te=scaler.transform(test[numeric_cols].fillna(med)).astype('float32')
    xc_tr=[]; xc_te=[]; cards=[]
    for col in categorical_cols:
        levels=sorted(pd.Series(train[col].dropna().unique()).tolist())
        mapping={v:i+1 for i,v in enumerate(levels)}
        xc_tr.append(train[col].map(mapping).fillna(0).astype(int).to_numpy())
        xc_te.append(test[col].map(mapping).fillna(0).astype(int).to_numpy())
        cards.append(len(levels)+1)
    return xn_tr,np.column_stack(xc_tr).astype('int64'),xn_te,np.column_stack(xc_te).astype('int64'),cards

In [ ]:
def fit_grouped_nam(df,features,numeric_cols,categorical_cols):
    y=df.severe_wasting.astype(int).to_numpy(); groups=df.hh1.to_numpy()
    sw=df.chweight.to_numpy().astype('float32'); sw=sw/sw.mean()
    cv=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_STATE)
    oof=np.zeros(len(df)); contrib=np.zeros((len(df),len(features))); folds=[]
    for fold,(tr,te) in enumerate(cv.split(df,y,groups),1):
        torch.manual_seed(RANDOM_STATE+fold)
        xntr,xctr,xnte,xcte,cards=prepare_fold(df.iloc[tr],df.iloc[te],numeric_cols,categorical_cols)
        model=NeuralAdditiveModel(len(numeric_cols),cards,HIDDEN)
        opt=torch.optim.AdamW(model.parameters(),lr=.01,weight_decay=WEIGHT_DECAY)
        Xn=torch.tensor(xntr); Xc=torch.tensor(xctr); yt=torch.tensor(y[tr],dtype=torch.float32)
        for _ in range(EPOCHS):
            model.train(); opt.zero_grad(); z=model(Xn,Xc)
            loss=nn.functional.binary_cross_entropy_with_logits(z,yt); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            z,C=model(torch.tensor(xnte),torch.tensor(xcte),True); p=torch.sigmoid(z).numpy(); C=C.numpy()
        oof[te]=p
        for j,name in enumerate(numeric_cols+categorical_cols): contrib[te,features.index(name)]=C[:,j]
        folds.append({'fold':fold,'weighted_auc':roc_auc_score(y[te],p,sample_weight=sw[te]),'unweighted_auc':roc_auc_score(y[te],p),'cases':int(y[te].sum())})
    metrics={'n':len(df),'cases':int(y.sum()),'weighted_prevalence':np.average(y,weights=sw),'weighted_roc_auc':roc_auc_score(y,oof,sample_weight=sw),'unweighted_roc_auc':roc_auc_score(y,oof),'weighted_pr_auc':average_precision_score(y,oof,sample_weight=sw),'weighted_brier':brier_score_loss(y,oof,sample_weight=sw)}
    centered=contrib-contrib.mean(0,keepdims=True); imp=np.mean(np.abs(centered),0)
    importance=pd.DataFrame({'feature':features,'mean_abs_logit_contribution':imp}); importance['share']=importance.mean_abs_logit_contribution/importance.mean_abs_logit_contribution.sum()
    return metrics,pd.DataFrame(folds),importance.sort_values('share',ascending=False).reset_index(drop=True)

metrics_all,folds_all,importance_all=fit_grouped_nam(df_all,features_all,numeric_all,categorical_all)
metrics_infant,folds_infant,importance_infant=fit_grouped_nam(df_infant,features_infant,numeric_infant,categorical_infant)
pd.DataFrame([{'analysis':'0-4 years',**metrics_all},{'analysis':'0-1 years',**metrics_infant}]).round(4)

In [ ]:
print('0-4 fold results')
display(folds_all.round(4))
print('0-4 contribution ranking')
display(importance_all.head(10).round(4))
print('0-1 fold results')
display(folds_infant.round(4))
print('0-1 contribution ranking')
display(importance_infant.head(12).round(4))

## Interpretation

The contribution ranking describes what the fitted neural network used; it is not an odds ratio, significance test, or causal effect. In the executed analysis used for `nn.md`, weighted out-of-fold ROC AUC was 0.492 for ages 0–4 and 0.522 for ages 0–1. Thus the neural additive models did not improve prediction over the logistic/tree benchmarks already in the repository. Rankings from such weak models should not be treated as new epidemiologic evidence.